# Visual Jev — per-second video runner

Helper notebook for `code/examples/video_per_second.py`. It runs the released
Qwen3-VL-4B + `visual-jev-4b-answer-sft` adapter over a video and asks the
dashcam question set against **each 1-second window**, writing one JSON per
second.

This notebook does **not** load the model itself — it shells out to the script,
so the script stays the single source of truth and you can run the same thing
from a plain terminal. The model is downloaded and loaded once per script run.

**Before running:** Runtime → Change runtime type → T4 GPU (or better).


In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Paths and settings

Edit these once. `REPO_DIR` is the repo root as mounted from Drive, matching
`colab_test_visual_jev.ipynb`.


In [ ]:
import os

REPO_DIR = "/content/drive/MyDrive/jev_dev"            # repo root, holds code/
SCRIPT   = f"{REPO_DIR}/code/examples/video_per_second.py"
OUT_DIR  = "/content/video_seconds"                    # per-second JSONs
FPS      = 2                                           # frames per 1-second window (1 = single frame)

assert os.path.exists(SCRIPT), f"script not found: {SCRIPT}\nclone or sync the repo into {REPO_DIR}"
print("script:", SCRIPT)
print("out dir:", OUT_DIR)


## Install dependencies

Mirrors the dependency step in `colab_test_visual_jev.ipynb`. Colab already
ships a CUDA torch, so only reinstall it if the `nvidia-smi` cell shows no GPU.
`opencv-python-headless` is what the script uses to read the video.


In [ ]:
!pip install -q -r {REPO_DIR}/code/requirements.txt opencv-python-headless
!pip install -U -q numpy torchao


## Choose the video

Default is the dashcam clip already in Drive (same path the other notebook
uses). Uncomment the upload block to bring in a different file.


In [ ]:
VIDEO_PATH = f"{REPO_DIR}/driving_cam.mp4"

# from google.colab import files
# uploaded = files.upload()
# VIDEO_PATH = next(iter(uploaded))

import os
assert os.path.exists(VIDEO_PATH), f"video not found: {VIDEO_PATH}"
print("video:", VIDEO_PATH, f"({os.path.getsize(VIDEO_PATH)} bytes)")


## Run the script over every second

`subprocess.run` streams the script's output live and raises if it fails. Add
`"--limit-seconds", "5"` to the command for a quick smoke test on a long video,
or set `"--fps", "1"` to score a single frame per second (image path) instead of
a multi-frame window.


In [ ]:
import subprocess, sys

cmd = [sys.executable, SCRIPT,
       "--video", VIDEO_PATH,
       "--out", OUT_DIR,
       "--device", "cuda",
       "--fps", str(FPS)]

print(" ".join(cmd), "\n")
subprocess.run(cmd, check=True)


## Read the results back

One `sec_XXXX.json` per second plus a combined `summary.json`.


In [ ]:
import glob, json
import pandas as pd

paths = sorted(glob.glob(f"{OUT_DIR}/sec_*.json"))
print(f"{len(paths)} per-second files in {OUT_DIR}")

records = [json.load(open(p)) for p in paths]

rows = []
for r in records:
    row = {"second": r["second"], "n_frames": r["n_frames"], "latency_s": r["latency_s"]}
    for qid, res in r["results"].items():
        row[qid] = res["prediction"]
        row[f"{qid}_p"] = round(max(res["probabilities"].values()), 3)
    rows.append(row)

df = pd.DataFrame(rows).set_index("second")
df


### One raw JSON file

The exact shape written per inference call.


In [ ]:
print(paths[0])
print(open(paths[0]).read())


### Decisions over time

Probability of each option, per second, for every question.


In [ ]:
import matplotlib.pyplot as plt

qids = list(records[0]["results"])
fig, axes = plt.subplots(len(qids), 1, figsize=(10, 2.6 * len(qids)), sharex=True)
if len(qids) == 1:
    axes = [axes]

for ax, qid in zip(axes, qids):
    options = list(records[0]["results"][qid]["probabilities"])
    seconds = [rec["second"] for rec in records]
    for option in options:
        ax.plot(seconds, [rec["results"][qid]["probabilities"][option] for rec in records],
                marker="o", label=option)
    ax.set_ylabel("P")
    ax.set_title(qid)
    ax.legend(fontsize=8, ncol=4)

axes[-1].set_xlabel("second")
plt.tight_layout()
plt.show()


## Notes

- The model loads once **per script run**, not per second — the per-second loop
  inside the script reuses it. Re-running the run cell re-downloads nothing but
  does reload the backbone into memory.
- `--fps 1` routes each second through the image path; `--fps > 1` passes the
  window's frames to the video path so the whole second is one shared context.
- The adapter was trained on still images, so per-second video decisions are out
  of distribution — check the probabilities against what is actually on screen
  before trusting them.
- Outputs land in `OUT_DIR`; point it at `f"{REPO_DIR}/out/video_seconds"` if you
  want them to persist in Drive.
